# Wrist RGB gripper opening on the BNC review clips

Generate the RGB observations with `main.py observe-gripper` using the command in `docs/KNOWROB_BASELINE.md`. This notebook compares those saved observations to the manual in-window reference. The eight targeted clips include only one endpoint with open jaws. Coverage and agreement describe this sample.

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

repo = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "main.py").is_file())
report_path = repo / "outputs/bnc_state_inspection/rgb_gripper_validation.json"
observations = json.loads(report_path.read_text(encoding="utf-8"))["observations"]

# Read the named snapshot blocks in the repository's manual reference file.
reference_path = repo / "knowledge/observations/bnc_validation_window.ttl"
reference = {}
for block in re.split(r"(?m)^observation:", reference_path.read_text(encoding="utf-8"))[1:]:
    name = block.split(maxsplit=1)[0]
    clip_id, endpoint = name.rsplit("_", 1)
    key = (clip_id, endpoint)
    assert key not in reference
    reference[key] = {
        **dict(re.findall(r'reassemble:(\w+) "([^"]+)"', block)),
        **dict(re.findall(r"reassemble:(\w+) reassemble:(\w+)", block)),
    }

In [ ]:
keys = [(row["clip_id"], row["endpoint"]) for row in observations]
assert len(keys) == len(set(keys)) == len(reference) and set(keys) == set(reference)

by_opening = Counter()
covered = correct = 0
for row, key in zip(observations, keys, strict=True):
    manual = reference[key]
    assert row["clip_id"] == manual["clipId"]
    assert row["recording_id"] == manual["recordingId"]
    assert str(row["segment_id"]) == manual["segmentId"]
    assert row["frame_index"] == int(manual["frameIndex"])
    assert row["observed_at"] == float(manual["observedAt"])
    expected = manual["gripperOpening"]
    predicted = row["gripper_opening"]
    by_opening[expected, predicted] += 1
    covered += predicted is not None
    correct += predicted == expected

print(f"Automatic opening coverage: {covered}/{len(keys)} endpoints")
print(f"Opening agreement on covered endpoints: {correct}/{covered}")
print("Manual opening -> RGB opening:", dict(by_opening))
print("Endpoints with manual socket engagement:",
      sum("socketEngagement" in row for row in reference.values()), "/", len(keys))
print("Endpoints with manual grip status:",
      sum("gripStatus" in row for row in reference.values()), "/", len(keys))